# Lab 1: Build a searchable company handbook

## Business mission

Employees need reliable answers from a company handbook. The handbook is a PDF, not a tiny text file. Your job is to turn that document into searchable records that still point back to the page where the evidence came from.

## What this lab builds

```text
company_handbook.pdf
  -> PDF loader
  -> pages with metadata
  -> recursive chunks
  -> embeddings
  -> Chroma vector store
  -> retrieved evidence
```

Run one cell at a time. Every code cell includes comments explaining the moving part and the production idea behind it.

## Exercise 1: Load the handbook PDF

**Mission:** Use a document loader to extract pages from a real PDF.

LangChain provides the loader. We are not manually searching for heading characters. The loader returns one document object per page and keeps page metadata for later citations.

In [ ]:
# Import the PDF loader supplied by LangChain.
# PyPDFLoader extracts selectable text and keeps page information.
from langchain_community.document_loaders import PyPDFLoader

# Keep the source path in one variable so the ingestion job can change it later.
PDF_PATH = "sample_documents/company_handbook.pdf"

# Create the loader, then read the PDF pages into memory.
loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print("Pages loaded:", len(pages))
print("First page metadata:", pages[0].metadata)
print(pages[0].page_content[:500])


### PDF extraction checkpoint

Confirm that the output contains readable text and page metadata. A scanned image-only PDF may return little or no text. In production, that case goes through OCR or a managed document service such as Amazon Textract before chunking.

## Exercise 2: Inspect the extracted pages

**Mission:** Look at the page objects before splitting them.

A page is already a structured document record. The text is in `page_content`; the source and page number are in `metadata`. This is the common shape that downstream chunking can use.

In [ ]:
# Inspect a few pages so we understand what the loader produced.
# We do not need to print the entire handbook.
for number, page in enumerate(pages[:3], start=1):
    print("\nPAGE", number)
    print("Metadata:", page.metadata)
    print(page.page_content[:300])


## Exercise 3: Split pages into retrieval chunks

**Mission:** Create smaller pieces that can be embedded and retrieved.

Recursive splitting tries paragraph and sentence boundaries before making a smaller cut. The starting values are 500 characters and 50 characters of overlap so the behavior is easy to inspect. Production teams measure these settings and may use token-based limits instead.

In [ ]:
# Import the splitter used by many LangChain RAG examples.
from langchain_text_splitters import RecursiveCharacterTextSplitter

# These are a starting configuration, not universal truth.
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

# The splitter tries larger boundaries first, such as paragraphs and sentences.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

# Split the page documents while retaining their source metadata.
chunks = splitter.split_documents(pages)

print("Chunks created:", len(chunks))


In [ ]:
# Print a few chunks so we can see what will become searchable rows.
# Notice that page metadata remains attached to each chunk.
for number, chunk in enumerate(chunks[:5]):
    print("\nCHUNK", number)
    print("Metadata:", chunk.metadata)
    print(chunk.page_content[:300])


### Chunking checkpoint

Ask whether each printed chunk is understandable by itself. If a rule is separated from its exception, the chunking configuration needs to be evaluated and changed.

## Exercise 4: Create embeddings and a local vector store

**Mission:** Convert the chunks into vectors and store them in Chroma.

Chroma is a local vector database that makes the retrieval mechanics visible. In production, the same index contract could use OpenSearch, pgvector, Pinecone, Qdrant, or a managed cloud search service.

In [ ]:
# OpenAIEmbeddings sends each chunk to the embedding model.
# The same model must embed both documents and future questions.
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

EMBEDDING_MODEL = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

# Chroma stores the chunk text, its metadata, and the vector representation.
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="company_handbook",
)

print("Indexed chunks:", len(chunks))


## Exercise 5: Retrieve evidence for a business question

**Mission:** Search the handbook before asking a language model to answer.

The model should receive retrieved evidence, not the entire PDF. We inspect the results first so we can verify the source pages and understand what the model will see.

In [ ]:
# This is the employee question our retrieval system must support.
QUESTION = "How many vacation days do employees receive?"
TOP_K = 3

# Chroma embeds the question and returns the closest chunks.
results = vector_store.similarity_search(QUESTION, k=TOP_K)

for number, result in enumerate(results, start=1):
    print("\nRESULT", number)
    print("Metadata:", result.metadata)
    print(result.page_content[:500])


## Exercise 6: Ask the language model using retrieved context

**Mission:** Give the model only the evidence found by the retriever and ask for a grounded answer.

The answer is only as reliable as the retrieved context. If the context does not contain the answer, the application should say that evidence is insufficient.

In [ ]:
# Build a compact context from the retrieved chunks.
context = "\n\n".join(result.page_content for result in results)

# Keep source metadata available for citations in a real application.
sources = [result.metadata for result in results]

# This prompt makes the grounding rule explicit.
prompt = f"""
Answer the question using only the context below.
If the context does not contain the answer, say that there is not enough evidence.

Context:
{context}

Question:
{QUESTION}
"""

from langchain_anthropic import ChatAnthropic
model = ChatAnthropic(model="claude-haiku-4-5")
answer = model.invoke(prompt)

print(answer.content)
print("Sources:", sources)


## Lab 1 checkpoint

You have now followed the production-shaped request path: a PDF loader produced page documents, a splitter created traceable chunks, an embedding model created vectors, Chroma stored the index, retrieval selected evidence, and an LLM answered from that evidence.

The local choices are replaceable. The pipeline contract is the important part.

## Production handoff: complex documents

This lab uses a text-based PDF so the loading and retrieval steps remain visible. A real corpus may also contain scanned PDFs, tables, images, audio, video, DOCX, HTML, and XML.

For those sources, the ingestion pipeline routes each format to the appropriate managed capability before chunking:

```text
scanned PDF or image -> OCR and layout extraction
PDF with tables       -> document analysis
DOCX or HTML          -> format-aware parser
audio or video        -> transcription and time-based chunks
                      -> normalized document records
                      -> chunking, embeddings, and indexing
```

On AWS, services such as Amazon Textract and Bedrock Data Automation can handle OCR, layout, tables, and multimodal extraction. The production team still validates the output, preserves page and source metadata, and quarantines failed documents. The local PDF loader is the learning implementation of the same ingestion stage.